In [2]:
!pip install -q langchain langchain-groq groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00


In [3]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

print("✅ Setup complete")
print(f"🤖 Model: llama-3.3-70b-versatile")

✅ Setup complete
🤖 Model: llama-3.3-70b-versatile


In [4]:
@tool
def check_customer_risk(balance: float, transactions: int, is_active: bool) -> str:
    """Check the risk level of a banking customer based on their profile.
    Use this when asked about customer risk assessment."""

    if not is_active:
        return "HIGH RISK — Account inactive"
    if balance < 1000:
        return f"HIGH RISK — Low balance: €{balance}"
    elif balance < 5000:
        return f"MEDIUM RISK — Moderate balance: €{balance}"
    else:
        return f"LOW RISK — Healthy balance: €{balance}"

print("✅ Tool 1 created: check_customer_risk")
print(f"   Name: {check_customer_risk.name}")
print(f"   Description: {check_customer_risk.description}")

✅ Tool 1 created: check_customer_risk
   Name: check_customer_risk
   Description: Check the risk level of a banking customer based on their profile.
    Use this when asked about customer risk assessment.


In [9]:
@tool
def check_aml_flag(transaction_amount: float, country: str) -> str:
    """Check if a transaction requires AML screening or has compliance flags.
    Use this when asked about AML, suspicious transactions or country risk."""

    high_risk_countries = ["Test1", "Test2", "Test3", "Test4"]
    flags = []

    if transaction_amount > 10000:
        flags.append("Transaction above EUR 10,000 — monitoring required")
    if transaction_amount > 5000:
        flags.append("Cash transaction above EUR 5,000 — documentation required")
    if country in high_risk_countries:
        flags.append(f"{country} is high risk jurisdiction — Enhanced Due Diligence required")

    if flags:
        return "🚨 AML FLAGS:\n" + "\n".join(flags)
    return "✅ No AML flags — transaction within normal parameters"


@tool
def check_credit_limit(client_exposure: float, total_portfolio: float, sector: str) -> str:
    """Check if a client exposure breaches credit risk concentration limits.
    Use this when asked about credit limits, exposure or portfolio concentration."""

    exposure_pct = (client_exposure / total_portfolio) * 100
    issues = []

    if exposure_pct > 10:
        issues.append(f"⚠️ Single client exposure {exposure_pct:.1f}% exceeds 10% limit")

    sector_limit = 30 if sector.lower() == "real estate" else 25
    if exposure_pct > sector_limit:
        issues.append(f"⚠️ Sector concentration exceeds {sector_limit}% limit for {sector}")

    if client_exposure > 5000000:
        issues.append("⚠️ Exposure above EUR 5M — immediate escalation required")

    if issues:
        return "LIMIT BREACH:\n" + "\n".join(issues)
    return f"✅ Exposure {exposure_pct:.1f}% — within all limits"


# All tools together
tools = [check_customer_risk, check_aml_flag, check_credit_limit]

print(f"✅ All {len(tools)} tools ready:")
for t in tools:
    print(f"  → {t.name}")

✅ All 3 tools ready:
  → check_customer_risk
  → check_aml_flag
  → check_credit_limit


In [10]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

system_message = """You are a senior banking compliance officer
at Northern Europe Bank.

You have access to these tools:
- check_customer_risk: assess customer risk level
- check_aml_flag: check AML compliance flags
- check_credit_limit: check credit exposure limits

Always use the appropriate tools to answer questions.
Think step by step and cite specific thresholds.
After using tools summarise your findings clearly."""

# Bind tools directly to the LLM
llm_with_tools = llm.bind_tools(tools)

print("✅ Agent ready")
print("🔧 Tools bound to LLM")
print("The LLM can now decide when to call each tool")

✅ Agent ready
🔧 Tools bound to LLM
The LLM can now decide when to call each tool


In [11]:
def run_agent(question):
    print(f"\n❓ Question: {question}")
    print("-" * 50)

    messages = [
        SystemMessage(content=system_message),
        HumanMessage(content=question)
    ]

    # Step 1 — LLM decides what to do
    response = llm_with_tools.invoke(messages)
    print(f"🤖 Agent thinking...")

    # Step 2 — Check if agent wants to use a tool
    if response.tool_calls:
        print(f"🔧 Agent decided to use tools:")

        for tool_call in response.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            print(f"   → Calling: {tool_name}")
            print(f"   → With args: {tool_args}")

            # Step 3 — Find and run the right tool
            for t in tools:
                if t.name == tool_name:
                    tool_result = t.invoke(tool_args)
                    print(f"   → Result: {tool_result}")

            # Step 4 — Add tool result to messages
            messages.append(response)
            from langchain_core.messages import ToolMessage
            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call['id']
                )
            )

        # Step 5 — LLM gives final answer using tool results
        final_response = llm_with_tools.invoke(messages)
        print(f"\n✅ FINAL ANSWER: {final_response.content}")

    else:
        # No tool needed — direct answer
        print(f"\n✅ ANSWER: {response.content}")

print("✅ Agent loop ready")

✅ Agent loop ready


In [8]:
# Test 1 — Customer Risk
run_agent("Check risk for customer with balance €750, 142 transactions, active account")


❓ Question: Check risk for customer with balance €750, 142 transactions, active account
--------------------------------------------------
🤖 Agent thinking...
🔧 Agent decided to use tools:
   → Calling: check_customer_risk
   → With args: {'balance': 750, 'is_active': True, 'transactions': 142}
   → Result: HIGH RISK — Low balance: €750.0

✅ FINAL ANSWER: The customer's risk level is high due to a low balance of €750.


In [12]:
# Test 2 — AML Check
run_agent("Is a €15,000 transaction from Test1 flagged for AML?")


❓ Question: Is a €15,000 transaction from Test1 flagged for AML?
--------------------------------------------------
🤖 Agent thinking...
🔧 Agent decided to use tools:
   → Calling: check_aml_flag
   → With args: {'country': 'Test1', 'transaction_amount': 15000}
   → Result: 🚨 AML FLAGS:
Transaction above EUR 10,000 — monitoring required
Cash transaction above EUR 5,000 — documentation required
Test1 is high risk jurisdiction — Enhanced Due Diligence required

✅ FINAL ANSWER: The €15,000 transaction from Test1 is flagged for AML due to the high-risk jurisdiction and the transaction amount exceeding the EUR 10,000 threshold, which requires monitoring. Additionally, since it's a cash transaction above EUR 5,000, documentation is required. Enhanced Due Diligence is also necessary because Test1 is a high-risk jurisdiction.


In [13]:
run_agent("Client exposure is €8M out of €50M portfolio in real estate sector — any breaches?")


❓ Question: Client exposure is €8M out of €50M portfolio in real estate sector — any breaches?
--------------------------------------------------
🤖 Agent thinking...
🔧 Agent decided to use tools:
   → Calling: check_credit_limit
   → With args: {'client_exposure': 8000000, 'sector': 'real estate', 'total_portfolio': 50000000}
   → Result: LIMIT BREACH:
⚠️ Single client exposure 16.0% exceeds 10% limit
⚠️ Exposure above EUR 5M — immediate escalation required

✅ FINAL ANSWER: Based on the check_credit_limit function, the client exposure of €8M out of a €50M portfolio in the real estate sector breaches the credit risk concentration limits. The single client exposure of 16.0% exceeds the 10% limit, and the exposure is above €5M, which requires immediate escalation.


In [14]:
run_agent("""
New client onboarding review:
- Account balance: €3,500
- Transactions: 89
- Account active: Yes
- Recent transaction: €12,000 from Test2
- Sector exposure: €6M out of €40M portfolio in manufacturing

Please do a complete compliance check.
""")


❓ Question: 
New client onboarding review:
- Account balance: €3,500
- Transactions: 89
- Account active: Yes
- Recent transaction: €12,000 from Test2
- Sector exposure: €6M out of €40M portfolio in manufacturing

Please do a complete compliance check.

--------------------------------------------------
🤖 Agent thinking...
🔧 Agent decided to use tools:
   → Calling: check_customer_risk
   → With args: {'balance': 3500, 'is_active': True, 'transactions': 89}
   → Result: MEDIUM RISK — Moderate balance: €3500.0
   → Calling: check_aml_flag
   → With args: {'country': 'Unknown', 'transaction_amount': 12000}
   → Result: 🚨 AML FLAGS:
Transaction above EUR 10,000 — monitoring required
Cash transaction above EUR 5,000 — documentation required
   → Calling: check_credit_limit
   → With args: {'client_exposure': 6000000, 'sector': 'manufacturing', 'total_portfolio': 40000000}
   → Result: LIMIT BREACH:
⚠️ Single client exposure 15.0% exceeds 10% limit
⚠️ Exposure above EUR 5M — immediate esca

In [ ]:
print("🏦 Banking Compliance Agent")
print("Ask about customer risk, AML or credit limits")
print("Type 'exit' to quit\n")

while True:
    question = input("❓ Your question: ")
    if question.lower() == 'exit':
        print("👋 Goodbye!")
        break
    run_agent(question)

🏦 Banking Compliance Agent
Ask about customer risk, AML or credit limits
Type 'exit' to quit

❓ Your question: a client in Test2 country

❓ Question: a client in Test2 country
--------------------------------------------------
🤖 Agent thinking...
🔧 Agent decided to use tools:
   → Calling: check_aml_flag
   → With args: {'country': 'Test2', 'transaction_amount': 10000}
   → Result: 🚨 AML FLAGS:
Cash transaction above EUR 5,000 — documentation required
Test2 is high risk jurisdiction — Enhanced Due Diligence required

✅ FINAL ANSWER: 
